# 00 - Data Overview

**Purpose:** confirm the dataset is loaded as expected and inspect the class
balance / a few samples before any model work. Executes against the split
directories built by `prepare-data` (or committed to a shared Drive folder).


## 1. Environment

In [ ]:
# In Colab with a GPU runtime:
# !pip install -e .            # install fauxtography + deps (see docs/setup.md)
import sys
assert "fauxtography" in sys.modules or True
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

DATA_ROOT = Path("data")   # <- point at your Drive path in Colab if needed
print("data root:", DATA_ROOT.resolve() if DATA_ROOT.exists() else "MISSING")


## 2. Split sizes and class balance

In [ ]:
from fauxtography.data.dataset import FauxtoDataset
from torchvision.transforms import ToTensor

for split in ("train", "val", "test"):
    p = DATA_ROOT / split
    if not p.exists():
        print(f"-- {split}: missing ({p})")
        continue
    ds = FauxtoDataset(p, transform=ToTensor())
    counts = {cls_: 0 for cls_ in ds.classes}
    for _, label in ds:
        counts[ds.classes[label]] += 1
    print(f"{split}: {len(ds)} images {counts}")


## 3. Sample grid (real vs fake)

In [ ]:
from torchvision.utils import make_grid

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for col, cls in enumerate(ds.classes):
    imgs = [ds[i][0] for i in range(len(ds)) if ds.classes[ds[i][1]] == cls][:8]
    grid = make_grid(imgs, nrow=4).permute(1, 2, 0).numpy()
    axes[col].imshow(np.clip(grid, 0, 1))
    axes[col].set_title(f"{cls} ({len(imgs)} samples)")
    axes[col].axis("off")
plt.tight_layout()
plt.show()
